In [0]:
import requests
import json
import io
import unicodedata
import pandas as pd
import re
import logging
from pyspark.sql.functions import current_timestamp, lit

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ── Dropbox OAuth2 credentials ────────────────────────────────────────────────
APP_KEY = dbutils.secrets.get(scope="ss2-bronze-layer", key="dropbox_app_key")
APP_SECRET = dbutils.secrets.get(scope="ss2-bronze-layer", key="dropbox_app_secret")
REFRESH_TOKEN = dbutils.secrets.get(scope="ss2-bronze-layer", key="dropbox_token")

# ── Pipeline configuration ────────────────────────────────────────────────────
WRITE_MODE    = "overwrite"   # "overwrite" or "append"
DROPBOX_PATH  = "/mspas"
TARGET_SCHEMA = "sandbox_bronze_ss2"  # must match the schema created in schema_creation.py
SOURCE_SYSTEM = "MSPAS"       # lineage tag appended to every ingested row


# ── Helpers ───────────────────────────────────────────────────────────────────

def get_dropbox_token(app_key, app_secret, refresh_token):
    # Access tokens expire after ~4 hours; refresh every run to avoid mid-job failures
    resp = requests.post(
        "https://api.dropbox.com/oauth2/token",
        data={
            "grant_type":    "refresh_token",
            "refresh_token": refresh_token,
            "client_id":     app_key,
            "client_secret": app_secret,
        },
    )
    data = resp.json()
    if "access_token" not in data:
        raise Exception(f"Dropbox auth failed: {data}")
    logger.info("Dropbox access token obtained.")
    return data["access_token"]


def clean_name(name):
    # Normalize table/column names to snake_case; strips .csv extension if present
    name = name.lower().replace(".csv", "")
    # ñ must be transliterated before NFKD decomposition (ñ → ni, so "año" → "anio")
    name = name.replace("ñ", "ni")
    # Strip diacritical marks from accented vowels (á→a, é→e, ó→o, ú→u, etc.)
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = re.sub(r'[^a-z0-9_]', '_', name)
    name = re.sub(r'_+', '_', name)
    return name.strip('_')


def list_csv_entries(token, root_path):
    # Walk the Dropbox folder tree and group CSV entries by their immediate parent folder
    resp = requests.post(
        "https://api.dropboxapi.com/2/files/list_folder",
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        data=json.dumps({"path": root_path, "recursive": True}),
    )
    result = resp.json()
    if "entries" not in result:
        raise Exception(f"Unexpected list_folder response: {result}")

    grouped = {}
    for entry in result["entries"]:
        if entry[".tag"] == "file" and entry["name"].lower().endswith(".csv"):
            folder = entry["path_display"].split("/")[-2]
            grouped.setdefault(folder, []).append(entry)
    return grouped


def download_csv(token, file_path):
    # Dropbox content endpoint requires the path in a request header, not the body
    resp = requests.post(
        "https://content.dropboxapi.com/2/files/download",
        headers={
            "Authorization":   f"Bearer {token}",
            "Dropbox-API-Arg": json.dumps({"path": file_path}),
        },
    )
    if resp.status_code != 200:
        raise Exception(f"HTTP {resp.status_code} for {file_path}")
    try:
        df = pd.read_csv(io.StringIO(resp.text), sep=None, engine="python")
    except Exception:
        # Fallback for files with malformed quoting (e.g. unescaped quotes mid-field)
        df = pd.read_csv(io.StringIO(resp.text), sep=None, engine="python", on_bad_lines="skip")
    # Log raw column names so schema drift across files is immediately visible in the job log
    logger.info(f"    Raw schema ({len(df.columns)} cols): {list(df.columns)}")
    # Normalize HERE, before returning, so pd.concat aligns on clean names.
    # Normalizing after concat causes phantom extra columns when two files spell the
    # same column differently (e.g. "Grupo Etario" vs "GrupoEtario").
    df.columns = dedupe_columns([clean_name(c) for c in df.columns])
    # Drop columns that pandas auto-generates from trailing delimiters (e.g. "Unnamed: 8")
    df = df.loc[:, ~df.columns.str.match(r'^unnamed_\d+$')]
    logger.info(f"    Clean schema ({len(df.columns)} cols): {list(df.columns)}")
    return df


def dedupe_columns(cols):
    # After normalization, distinct source columns can collide (e.g. "CIE-10" and "CIE_10" → "cie_10")
    seen = {}
    result = []
    for col in cols:
        count = seen.get(col, 0)
        seen[col] = count + 1
        result.append(col if count == 0 else f"{col}_{count + 1}")
    return result


def ingest_folder(token, folder_name, entries, schema, write_mode, source_system):
    table_name = f"{schema}.dbx_{clean_name(folder_name)}"
    logger.info("-" * 60)
    logger.info(f"Source folder : {folder_name}")
    logger.info(f"Target table  : {table_name}  [mode={write_mode}, files={len(entries)}]")

    frames = []
    for entry in entries:
        try:
            logger.info(f"  Downloading: {entry['name']}")
            frames.append(download_csv(token, entry["path_display"]))
        except Exception as exc:
            # Log and skip so one corrupt file doesn't abort the entire folder batch
            logger.warning(f"  Skipped {entry['name']}: {exc}")

    if not frames:
        logger.warning(f"  No valid data for '{folder_name}', skipping write.")
        return

    combined = pd.concat(frames, ignore_index=True)
    # Columns are already normalized per-file; dedupe is a safety net for any residual collision
    combined.columns = dedupe_columns(list(combined.columns))

    # Resolve object columns with mixed types (int rows + str rows) that Arrow cannot infer.
    # errors='coerce' turns non-numeric values into NaN so we can measure the numeric ratio.
    for col in combined.select_dtypes(include="object").columns:
        coerced = pd.to_numeric(combined[col], errors="coerce")
        non_null = combined[col].notna().sum()
        if non_null > 0 and coerced.notna().sum() / non_null >= 0.9:
            # Column is predominantly numeric → keep as float (NaN-compatible)
            combined[col] = coerced
        else:
            # Text column → cast non-null values to str so Arrow maps to StringType uniformly
            combined[col] = combined[col].where(combined[col].isna(), combined[col].astype(str))

    sdf = spark.createDataFrame(combined)
    sdf = (sdf
           .withColumn("ingestion_timestamp", current_timestamp())
           .withColumn("source_system", lit(source_system)))

    (sdf.write
        .format("delta")
        .mode(write_mode)
        .option("mergeSchema", "true")   # allows new columns across runs without schema errors
        .saveAsTable(table_name))

    logger.info(f"  Wrote {len(combined):,} rows → {table_name}")


# ── Entry point ───────────────────────────────────────────────────────────────
try:
    token   = get_dropbox_token(APP_KEY, APP_SECRET, REFRESH_TOKEN)
    folders = list_csv_entries(token, DROPBOX_PATH)

    for folder_name, entries in folders.items():
        ingest_folder(token, folder_name, entries, TARGET_SCHEMA, WRITE_MODE, SOURCE_SYSTEM)

    logger.info("Pipeline completed successfully.")
except Exception as e:
    logger.error(f"Pipeline failed: {e}")
    raise  # re-raise so the Databricks job status reflects the failure
